In [ ]:
!pip install demucs pydub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 15.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 7.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.3/249.3 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 8.4 MB/s eta 0:00:00
  Created wheel for demucs: filename=demucs-4.0.1-py3-none-any.whl size=78388 sha256=76a0dfea8f85ada0a648667715528870ebb66776196c30580ce7a0e5a6459ff1
  Stored in directory: /root/.cache/pip/wheels/1b/0c/20/a3b3daa1f9b65c8b0445729f94740ec335d0f86f1066c5c414
  Created wheel for dora-search: filename=dora_search-0.1.12-py3-none-any.whl size=75195 sha256=7f14ad3cb9b7af95b18efd3e3e50816f59d0df0d09180d992951b556a7571e98
  Stored i

In [ ]:
# ==========================================
# 1. INSTALL ALL REQUIRED PACKAGES
# ==========================================
print("Installing packages... This might take a minute.")
!pip install demucs pydub torch

import os
import subprocess
import torch
from pydub import AudioSegment
import IPython.display as ipd

# ==========================================
# 2. DEFINE THE CLEANING FUNCTION
# ==========================================
def clean_audio_with_demucs(input_path, output_filename="cleaned_speech.wav"):
    """
    Cleans any generic audio file (M4A, MP3, WAV, etc.) using Meta's Demucs.
    Automatically handles formatting, GPU acceleration, and folder organization.
    """
    if not os.path.exists(input_path):
        print(f"❌ Error: Input file '{input_path}' not found. Please check your path and upload the file.")
        return None

    print(f"\n--- Processing File: {os.path.basename(input_path)} ---")

    # Step A: Standardize input to a temporary WAV file for stable processing
    print("[1/3] Standardizing audio format...")
    temp_input = "temp_input_demucs.wav"
    audio = AudioSegment.from_file(input_path)
    audio.export(temp_input, format="wav")

    # Step B: Automatically detect hardware acceleration (T4 GPU vs CPU)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[2/3] Running Demucs AI Speech Model on system: [{device.upper()}]...")

    # Build the Demucs CLI command using the speech-specific telephony model
    command = [
        "demucs",
        "-n", "htdemucs_telephony",
        "-d", device,
        temp_input
    ]

    # Run Demucs via shell command
    result = subprocess.run(command, capture_output=True, text=True)

    if result.returncode != 0:
        print("❌ Error during Demucs processing:")
        print(result.stderr)
        return None

    # Step C: Extract the final file and clean up temporary directory artifacts
    print("[3/3] Finalizing and cleaning up directory...")

    # Demucs saves output in nested folders: separated/htdemucs_telephony/temp_input_demucs/vocals.wav
    expected_output_path = os.path.join("separated", "htdemucs_telephony", "temp_input_demucs", "vocals.wav")

    if os.path.exists(expected_output_path):
        # Move and rename the final file to your specified output filename
        if os.path.exists(output_filename):
            os.remove(output_filename)
        os.rename(expected_output_path, output_filename)
        print(f"🎉 Success! Cleaned audio saved as: {output_filename}")
    else:
        print("❌ Error: Could not find the expected Demucs output file.")
        output_filename = None

    # Housekeeping: Clean up processing artifacts
    if os.path.exists(temp_input):
        os.remove(temp_input)
    subprocess.run(["rm", "-rf", "separated"])

    return output_filename

# ==========================================
# 3. EXECUTION BLOCK (CHANGE YOUR PATH HERE)
# ==========================================

# ⬇️ CHANGE THIS LINE to your uploaded audio file path (M4A, MP3, WAV, etc.)
YOUR_AUDIO_PATH = '/content/AUD-20260402-WA0018.m4a'

# Name your clean output file
OUTPUT_FILE = 'perfect_cleaned_audio.wav'

# Run the pipeline
final_output = clean_audio_with_demucs(YOUR_AUDIO_PATH, output_filename=OUTPUT_FILE)

# Play the output directly in your browser if successful
if final_output and os.path.exists(final_output):
    print("\n🎧 Listen to Cleaned Audio:")
    ipd.display(ipd.Audio(final_output))

Installing packages... This might take a minute.

--- Processing File: AUD-20260402-WA0018.m4a ---
[1/3] Standardizing audio format...
[2/3] Running Demucs AI Speech Model on system: [CUDA]...
❌ Error during Demucs processing:
FATAL: htdemucs_telephony is neither a single pre-trained model or a bag of models.



In [ ]:
# ==========================================
# 1. INSTALL ALL REQUIRED PACKAGES
# ==========================================
print("Installing packages... This might take a minute.")
# torchaudio and speechbrain handle the AI model, pydub handles general file formats
!pip install speechbrain torchaudio pydub torch

import os
import torch
import torchaudio
from pydub import AudioSegment
import IPython.display as ipd
from speechbrain.inference.enhancement import SpectralMaskEnhancement

# ==========================================
# 2. DEFINE THE OPTION B CLEANING FUNCTION
# ==========================================
def clean_audio_with_metricgan(input_path, output_filename="cleaned_speech_b.wav"):
    """
    Cleans any generic audio file (M4A, MP3, WAV, etc.) using Microsoft's MetricGAN+.
    Automatically handles hardware acceleration, downsampling, and format standardization.
    """
    if not os.path.exists(input_path):
        print(f"❌ Error: Input file '{input_path}' not found. Please check your path.")
        return None

    print(f"\n--- Processing File: {os.path.basename(input_path)} ---")

    # Step A: Standardize input format and resample to 16kHz
    # SpeechBrain models are strictly trained on 16kHz mono audio.
    print("[1/3] Standardizing format and converting to 16kHz...")
    temp_16k_wav = "temp_16kHz_input.wav"
    audio = AudioSegment.from_file(input_path)
    audio = audio.set_frame_rate(16000).set_channels(1)
    audio.export(temp_16k_wav, format="wav")

    # Step B: Automatically detect hardware acceleration (T4 GPU vs CPU)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[2/3] Loading MetricGAN+ AI Model on [{device.upper()}]...")

    # Load pre-trained model from SpeechBrain Hub
    enhance_model = SpectralMaskEnhancement.from_hparams(
        source="speechbrain/metricgan-plus-voicebank",
        savedir="pretrained_models/metricgan-plus",
        run_opts={"device": device}
    )

    # Step C: Process the audio waveform through the neural network
    print("[3/3] Running deep noise suppression neural network...")
    enhanced_tensor = enhance_model.enhance_file(temp_16k_wav)

    # Save the resulting tensor back into your final output file
    if os.path.exists(output_filename):
        os.remove(output_filename)

    torchaudio.save(output_filename, enhanced_tensor.cpu(), 16000)
    print(f"🎉 Success! Cleaned audio saved as: {output_filename}")

    # Housekeeping: Remove the temporary file
    if os.path.exists(temp_16k_wav):
        os.remove(temp_16k_wav)

    return output_filename

# ==========================================
# 3. EXECUTION BLOCK (CHANGE YOUR PATH HERE)
# ==========================================

# ⬇️ CHANGE THIS LINE to your uploaded audio file path (M4A, MP3, WAV, etc.)
YOUR_AUDIO_PATH = '/content/AUD-20260402-WA0018.m4a'

# Name your clean output file
OUTPUT_FILE = 'metricgan_cleaned_audio.wav'

# Run the pipeline
final_output = clean_audio_with_metricgan(YOUR_AUDIO_PATH, output_filename=OUTPUT_FILE)

# Play the output directly in your browser if successful
if final_output and os.path.exists(final_output):
    print("\n🎧 Listen to Cleaned Audio (Option B):")
    ipd.display(ipd.Audio(final_output))